In [ ]:
# ex. 1
import random
import pandas as pd

def alege_studenti(cale_fisier, n, are_antet=False):
    # citim csv-ul; daca primul rand e antet, il sarim cu header=0
    df = pd.read_csv(cale_fisier, header=0 if are_antet else None)
    # luam prima coloana (numele studentilor) ca lista, fara valori lipsa
    lista = df.iloc[:, 0].dropna().tolist()
    # random.sample alege n elemente fara repetitie (da eroare daca n > len(lista))
    return random.sample(lista, n)

# exemplu de test cu nume inventate: stergeti liniile astea si folositi fisierul vostru
pd.DataFrame({"nume": [f"Student {i}" for i in range(1, 21)]}).to_csv("studenti.csv", index=False)
print(alege_studenti("studenti.csv", 5, are_antet=True))

**Ex. 2 a)** La fiecare pas moneda da stema cu probabilitatea $p = 0.5$, independent de pasii anteriori, iar jocul se opreste la prima stema. Deci $N$ este numarul de aruncari pana la prima stema, adica $N$ urmeaza o distributie **geometrica** de parametru $p = 0.5$:

$$P(N = n) = (1-p)^{n-1}\, p = \left(\tfrac12\right)^n, \quad n = 1, 2, 3, \dots$$

cu media $E[N] = 1/p = 2$.

In [ ]:
# ex. 2 b)
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng()

def joaca(p=0.5):
    # simuleaza un joc; intoarce (N, S)
    n = 0      # numarul de pasi
    s = 0.0    # suma pe care o da al doilea jucator
    while True:
        n += 1
        if rng.random() < p:
            # stema: se arunca zarul (1..6), se plateste z - 3 si jocul se opreste
            z = rng.integers(1, 7)
            s += z - 3
            break
        else:
            # ban: al doilea jucator da 0.5 $ si jocul continua
            s += 0.5
    return n, s

print("N, S =", joaca())

In [ ]:
# ex. 2 c) si d)
def simuleaza(p, n_jocuri=10000):
    # ruleaza n_jocuri jocuri si intoarce vectorii cu valorile lui N si S
    rezultate = [joaca(p) for _ in range(n_jocuri)]
    N = np.array([r[0] for r in rezultate])
    S = np.array([r[1] for r in rezultate])
    return N, S

def histograma(S, p):
    # S ia valori multiple de 0.5, deci folosim bin-uri de latime 0.5 centrate pe ele
    bins = np.arange(S.min() - 0.25, S.max() + 0.75, 0.5)
    plt.hist(S, bins=bins, density=True, edgecolor="black")
    plt.axvline(S.mean(), color="red", linestyle="--", label=f"media = {S.mean():.2f}")
    plt.title(f"distributia lui S (p = {p})")
    plt.xlabel("S")
    plt.ylabel("frecventa relativa")
    plt.legend()
    plt.show()

# p = 0.5 este moneda corecta (c), iar 0.3 si 0.7 sunt monedele masluite (d)
for p in [0.5, 0.3, 0.7]:
    N, S = simuleaza(p)
    # S = 0.5 * (N - 1) + (Z - 3), deci E[S] = 0.5 * (1 - p) / p + 0.5
    print(f"p = {p}: media lui S simulata = {S.mean():.3f}, teoretica = {0.5 * (1 - p) / p + 0.5:.3f}")
    histograma(S, p)
# cu p mic stema pica rar, jocul dureaza mai mult si se aduna mai multe plati de 0.5, deci S creste

**Ex. 3 (de ce?)** Frizerii taie in medie 3, 6 si 4 clienti pe ora, deci impreuna deservesc $3 + 6 + 4 = 13$ clienti pe ora. Pe termen lung, un client ales la intamplare a fost deservit de frizerul $i$ cu probabilitatea egala cu partea lui din total, $\lambda_i / \sum_j \lambda_j$, adica $3/13$, $6/13$ si $4/13$. Deci $X$ este un amestec de exponentiale, cu densitatea $f(x) = \sum_i p_i \lambda_i e^{-\lambda_i x}$.

In [ ]:
# ex. 3
import arviz as az

lambdas = np.array([3, 6, 4])       # clienti pe ora, pentru fiecare frizer
probs = lambdas / lambdas.sum()     # 3/13, 6/13, 4/13
n = 10000

# pentru fiecare client alegem frizerul, apoi timpul de servire exponential cu rata lui
frizer = rng.choice(3, size=n, p=probs)
X = rng.exponential(scale=1 / lambdas[frizer])

print("media lui X (estimata):", X.mean())
print("deviatia standard a lui X (estimata):", X.std())
# teoretic: E[X] = suma p_i / lambda_i = 3/13
print("media teoretica:", np.sum(probs / lambdas))

# densitatea aproximata cu kde, comparata cu densitatea exacta a amestecului
x = np.linspace(0, X.max(), 500)
try:
    az.plot_kde(X, label="kde (simulare)")
except AttributeError:
    # in arviz 1.x plot_kde nu mai exista, folosim kde din scipy
    from scipy.stats import gaussian_kde
    plt.plot(x, gaussian_kde(X)(x), label="kde (simulare)")
f = sum(pi * li * np.exp(-li * x) for pi, li in zip(probs, lambdas))
plt.plot(x, f, "r--", label="densitate exacta")
plt.xlabel("timp de servire (ore)")
plt.legend()
plt.show()